# Applications of Matrix Multiplication (Bilingual Lab)
## Aplicaciones de la multiplicación de matrices (Laboratorio bilingüe)

**Goal / Meta:** See matrices *do real work*: encryption/decryption and photo filters.  
**Objetivo / Meta:** Ver cómo las matrices *hacen trabajo real*: encriptación/desencriptación y filtros de fotos.

**You will learn / Vas a aprender:**
- A matrix can **transform information** (letters → secret code → letters).  
- Una matriz puede **transformar información** (letras → código secreto → letras).
- A matrix can **transform images** (pixel colors).  
- Una matriz puede **transformar imágenes** (colores de pixeles).
- The inverse matrix is an **undo** button (when it exists).  
- La matriz inversa es un botón de **deshacer** (cuando existe).


## Part A: Cryptography (Hill Cipher idea)
## Parte A: Criptografía (idea del cifrado de Hill)

**Big idea / Idea grande:**  
Encryption applies a matrix. Decryption applies the inverse matrix.  
La encriptación aplica una matriz. La desencriptación aplica la matriz inversa.

**Important note / Nota importante:**  
Real cryptography often uses “wrap-around” arithmetic (like a clock). We will do the same so numbers stay in a letter range.  
La criptografía real a menudo usa aritmética “circular” (como un reloj). Haremos lo mismo para que los números se queden dentro del rango de letras.

**STOP and think / ALTO y piensa:**  
Why would “wrap-around” be useful for letters?  
¿Por qué sería útil la aritmética circular para letras?


In [ ]:
# Run this cell first.
# Ejecuta esta celda primero.

import numpy as np
import string
from math import gcd

alphabet = string.ascii_uppercase  # 'A'...'Z'
char_to_num = {ch:i for i,ch in enumerate(alphabet)}   # A->0, ..., Z->25
num_to_char = {i:ch for i,ch in enumerate(alphabet)}   # 0->A, ..., 25->Z

def clean_text(s):
    """Keep letters only, make uppercase.
    Mantiene solo letras y pone en MAYÚSCULAS."""
    s = s.upper()
    return ''.join([ch for ch in s if ch in alphabet])

def text_to_nums(s):
    s = clean_text(s)
    return [char_to_num[ch] for ch in s]

def nums_to_text(nums):
    return ''.join(num_to_char[int(n)%26] for n in nums)

def mod_inv(a, m):
    """Modular inverse of a mod m (find x so a*x ≡ 1 mod m).
    Inverso modular de a mod m (encuentra x tal que a*x ≡ 1 mod m)."""
    a = a % m
    # brute force is fine here because m=26 is small
    for x in range(m):
        if (a * x) % m == 1:
            return x
    return None

def matrix_mod_inv_2x2(A, m=26):
    """Inverse of 2x2 matrix modulo m, if it exists.
    Inversa de una matriz 2x2 módulo m, si existe."""
    a,b = int(A[0,0]), int(A[0,1])
    c,d = int(A[1,0]), int(A[1,1])
    det = (a*d - b*c) % m
    inv_det = mod_inv(det, m)
    if inv_det is None:
        return None, det
    adj = np.array([[d, -b],
                    [-c, a]], dtype=int)
    Ainv = (inv_det * adj) % m
    return Ainv, det

def hill_encrypt(text, A):
    nums = text_to_nums(text)
    # pad with X (23) if odd length / rellena con X si la longitud es impar
    if len(nums) % 2 == 1:
        nums.append(char_to_num['X'])
    out = []
    for i in range(0, len(nums), 2):
        v = np.array([[nums[i]],[nums[i+1]]], dtype=int)
        w = (A @ v) % 26
        out.extend([int(w[0,0]), int(w[1,0])])
    return nums_to_text(out)

def hill_decrypt(cipher, Ainv):
    nums = text_to_nums(cipher)
    if len(nums) % 2 == 1:
        nums.append(char_to_num['X'])
    out = []
    for i in range(0, len(nums), 2):
        v = np.array([[nums[i]],[nums[i+1]]], dtype=int)
        w = (Ainv @ v) % 26
        out.extend([int(w[0,0]), int(w[1,0])])
    return nums_to_text(out)

print("Ready. / Listo.")

### Choose an encryption matrix (2×2)
### Elige una matriz de encriptación (2×2)

Use the matrix below (start here), or try your own later.  
Usa la matriz de abajo (empieza aquí), o prueba la tuya después.

**Rule / Regla:** The matrix must be invertible (undo-able).  
La matriz debe ser invertible (se puede deshacer).

**Hint / Pista:** If the code says there is no inverse, try a different matrix.  
Si el código dice que no hay inversa, prueba otra matriz.


In [ ]:
# Encryption matrix A (2x2)
# Matriz de encriptación A (2x2)

A = np.array([[3, 3],
              [2, 5]], dtype=int)

Ainv, det = matrix_mod_inv_2x2(A, m=26)

print("A =\n", A)
print("det(A) mod 26 =", det)
print("Inverse exists? / ¿Existe inversa?", Ainv is not None)
print("A^{-1} mod 26 =\n", Ainv)

### Try encrypting and decrypting a message
### Prueba encriptar y desencriptar un mensaje

Type a short message. Letters only work best (A–Z).  
Escribe un mensaje corto. Funciona mejor con letras (A–Z).

**STOP / ALTO:** Predict: will the decrypted message match the original? Why?  
Predice: ¿la desencriptación va a coincidir con el mensaje original? ¿Por qué?


In [ ]:
# Type your message here / Escribe tu mensaje aquí
message = "MATH IS POWER"

cipher = hill_encrypt(message, A)
plain  = hill_decrypt(cipher, Ainv)

print("Original / Original:", clean_text(message))
print("Encrypted / Encriptado:", cipher)
print("Decrypted / Desencriptado:", plain)

### Explore: What if the matrix cannot be undone?
### Explora: ¿Qué pasa si la matriz no se puede deshacer?

Try a matrix that *collapses* information (for example, two rows that are multiples).  
Prueba una matriz que *colapsa* información (por ejemplo, dos filas que son múltiplos).

**Question / Pregunta:** What happens to decryption when there is no inverse?  
¿Qué pasa con la desencriptación cuando no hay inversa?


In [ ]:
# Try a "bad" matrix (often not invertible)
# Prueba una matriz "mala" (a menudo no es invertible)

B = np.array([[2, 4],
              [1, 2]], dtype=int)

Binv, detB = matrix_mod_inv_2x2(B, m=26)

print("B =\n", B)
print("det(B) mod 26 =", detB)
print("Inverse exists? / ¿Existe inversa?", Binv is not None)
print("B^{-1} mod 26 =\n", Binv)

## Part B: Photo Filters (Color Transformations)
## Parte B: Filtros de fotos (Transformaciones de color)

**Big idea / Idea grande:**  
An image is a grid of pixels. Each pixel has numbers for color (R, G, B).  
Una imagen es una cuadrícula de pixeles. Cada pixel tiene números de color (R, G, B).

We can transform the colors using a 3×3 matrix.  
Podemos transformar los colores con una matriz 3×3.

**Important note / Nota importante:**  
If values go outside the valid range, we will **clip** them (cap at the min/max).  
Si los valores salen del rango válido, los vamos a **recortar** (poner un límite).


### Step 1: Upload (or use a provided image)
### Paso 1: Subir una imagen (o usar una imagen proporcionada)

**If your teacher provided images:** use the file name shown in the file list (left panel).  
**Si tu profe dio imágenes:** usa el nombre del archivo en la lista de archivos (panel izquierdo).

**If you upload your own image:**  
1) Click the file browser (left) → Upload  
2) Upload a small JPG/PNG (not huge)  
3) Set `img_path` to your file name

**Si subes tu propia imagen:**  
1) Haz clic en el explorador (izquierda) → Upload  
2) Sube un JPG/PNG pequeño (no enorme)  
3) Cambia `img_path` al nombre de tu archivo


In [ ]:
# Choose the image file path here.
# Elige la ruta/nombre del archivo aquí.

img_path = "images/city.png"   # example / ejemplo: "myphoto.jpg"

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

img = mpimg.imread(img_path)

plt.figure(figsize=(5,5))
plt.title("Original / Original")
plt.axis("off")
plt.imshow(img)
plt.show()

print("Shape:", img.shape, "dtype:", img.dtype, "min/max:", float(np.min(img)), float(np.max(img)))

### Step 2: Choose a color matrix (3×3)
### Paso 2: Elige una matriz de color (3×3)

Start with Identity (no change). Then try one suggested matrix.  
Empieza con la Identidad (sin cambio). Luego prueba una matriz sugerida.

**Tip / Consejo:** Change ONE number at a time and re-run.  
Cambia SOLO UN número a la vez y vuelve a ejecutar.


In [ ]:
# Color transform matrix M (3x3)
# Matriz de transformación de color M (3x3)

# Identity (no change) / Identidad (sin cambio)
M = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])

# Try ONE of these by replacing M above:
# Prueba UNA de estas reemplazando M arriba:

# 1) Swap Red and Green / Intercambiar Rojo y Verde
# M = np.array([[0,1,0],[1,0,0],[0,0,1]], dtype=float)

# 2) Remove Blue / Quitar Azul
# M = np.array([[1,0,0],[0,1,0],[0,0,0]], dtype=float)

# 3) Boost brightness a bit / Subir brillo un poco
# M = np.array([[1.2,0,0],[0,1.2,0],[0,0,1.2]], dtype=float)

print("M =\n", M)

In [ ]:
# Apply matrix filter to the image (RGB).
# Aplicar el filtro de matriz a la imagen (RGB).

img_float = img.astype(float)

# Normalize to 0..1 if needed / Normaliza a 0..1 si es necesario
if img_float.max() > 1.5:
    img_float = img_float / 255.0

# Handle alpha channel / Manejar canal alfa (transparencia)
has_alpha = (img_float.shape[-1] == 4)
if has_alpha:
    rgb = img_float[..., :3]
    alpha = img_float[..., 3:4]
else:
    rgb = img_float[..., :3]
    alpha = None

# Apply transform: each pixel is a (R,G,B) vector
# Aplica transformación: cada pixel es un vector (R,G,B)
out_rgb = rgb @ M.T

# Clip to valid range 0..1 / Recortar al rango válido 0..1
out_rgb = np.clip(out_rgb, 0.0, 1.0)

# Reattach alpha if needed / Re-agregar alfa si existe
if has_alpha:
    out = np.concatenate([out_rgb, alpha], axis=-1)
else:
    out = out_rgb

plt.figure(figsize=(5,5))
plt.title("Filtered / Filtrada")
plt.axis("off")
plt.imshow(out)
plt.show()

### Reflection / Reflexión (write in complete sentences)

1) **Cipher:** Why does the inverse matrix “undo” the encryption?  
   **Cifrado:** ¿Por qué la matriz inversa “deshace” la encriptación?

2) **Cipher:** What happened when there was no inverse matrix?  
   **Cifrado:** ¿Qué pasó cuando no había matriz inversa?

3) **Photo:** Describe one change you made to the matrix and what it did to the image.  
   **Foto:** Describe un cambio que hiciste a la matriz y qué le hizo a la imagen.

4) **Big idea:** How are the cipher and the photo filter similar?  
   **Idea grande:** ¿Cómo se parecen el cifrado y el filtro de foto?
